# NB4b — U-Net Baseline (CNN cùng họ DL)

Baseline đối chứng cho GraphCast-SEA: **cùng dữ liệu memmap, cùng loss latitude-weighted, cùng residual ΔX, cùng vòng đánh giá rollout** — chỉ thay lõi GNN bằng U-Net 2D trên lưới đều.

Mục đích là so sánh công bằng, KHÔNG phải để U-Net thắng; giữ ở mức base nhỏ.

In [1]:
!pip install torch zarr xarray dask numpy scipy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 92.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.

In [2]:
import os, json, math
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # giảm phân mảnh VRAM
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import xarray as xr

# ── Paths ─────────────────────────────────────────────────────────
STATS_JSON = "/kaggle/input/notebooks/phongngtun/stat-compute/era5_normalization_stats.json"
OUT      = "/kaggle/working"

# Thư mục SCRATCH để chứa memmap fp16 (KHÔNG tính vào 20GB output của Kaggle).
# /kaggle/tmp tồn tại trên Kaggle; fallback /tmp nếu chạy nơi khác.
SCRATCH = "/kaggle/tmp/cache_mm" if os.path.isdir("/kaggle") else "/tmp/cache_mm"
os.makedirs(SCRATCH, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = False     # autotune conv/scatter kernels
torch.backends.cuda.matmul.allow_tf32 = True   # TF32 trên Ampere+ (vô hại trên T4/P100)
print(f"✓ Device: {DEVICE}")

# ── Cấu hình Năm ──────────────────────────────────────────────────
TRAIN_YEARS = [2020, 2021, 2022, 2023, 2024]
VAL_YEARS   = [2025]
TEST_YEARS  = [2026]

# ── Biến static (chỉ làm input) ───────────────────────────────────
STATIC_VARS  = ['lsm', 'geopotential_surf']

# ── Hyperparams ───────────────────────────────────────────────────
HIDDEN_DIM   = 128
N_LAYERS     = 8           # ↑4→8: processor chạy trên mesh ~500 node nên gần như miễn phí
BATCH_SIZE   = 12          # global; DataParallel chia 12 → 6/GPU khi T4×2 (sát ngưỡng an toàn). 1 GPU: hạ về 4-6.
LR           = 1.5e-4      # ↑ theo linear scaling khi batch 8→12 (×1.5). Nếu loss đầu epoch nhảy loạn → hạ về 1e-4
N_EPOCHS     = 50
PATIENCE     = 7
MESH_RES     = 2.0

# ── U-Net baseline ────────────────────────────────────────────────
UNET_BASE    = 48          # số kênh gốc (chia hết cho 8 vì GroupNorm)
UNET_DEPTH   = 3           # số tầng down/up (159×201 → pad bội số 2^3=8)

# ── Tối ưu I/O & tốc độ ───────────────────────────────────────────
USE_AMP      = True        # mixed precision (fp16) — nhanh hơn, ít VRAM hơn
USE_MULTI_GPU= True        # dùng cả 2 GPU (T4×2) qua DataParallel nếu có
NUM_WORKERS  = 4           # đọc memmap song song (memmap fork-safe trên Linux)
PREFETCH     = 4
MM_BLOCK     = 100         # đọc Zarr theo khối 100 bước = khớp chunk gốc {'time':100}

print(f"  U-Net base : {UNET_BASE} | depth : {UNET_DEPTH}")
print(f"  Batch size : {BATCH_SIZE} | Epochs : {N_EPOCHS} | AMP : {USE_AMP}")
print(f"  Workers    : {NUM_WORKERS} | Scratch : {SCRATCH}")
print(f"  Static vars: {STATIC_VARS}")


✓ Device: cuda
  U-Net base : 48 | depth : 3
  Batch size : 12 | Epochs : 50 | AMP : True
  Workers    : 4 | Scratch : /kaggle/tmp/cache_mm
  Static vars: ['lsm', 'geopotential_surf']


## 4.1 Load Data & Build Graph

In [3]:
import os

print("--- Load multi-year data (lazy & auto-search Kaggle input) ---")

def find_zarr_path(folder_name):
    """
    Quét sâu (deep scan) toàn bộ thư mục /kaggle/input để tìm chính xác
    thư mục Zarr, bất chấp việc Kaggle có tạo thêm thư mục con hay không.
    """
    for root, dirs, files in os.walk('/kaggle/input'):
        if folder_name in dirs:
            return os.path.join(root, folder_name)
            
    raise FileNotFoundError(f"❌ Không tìm thấy: {folder_name}. Hãy kiểm tra xem bạn đã add đủ input chưa!")

def load_split(years):
    xs, labels, eyes = [], [], []
    for y in years:
        # Tự động quét đường dẫn thực tế trên đĩa Kaggle
        path_x     = find_zarr_path(f"brain_{y}_x.zarr")
        path_label = find_zarr_path(f"brain_{y}_label.zarr")
        path_eye   = find_zarr_path(f"eye_{y}_norm.zarr")
        
        # Mở Zarr lười biếng (lazy loading)
        xs.append(xr.open_zarr(path_x, consolidated=True))
        labels.append(xr.open_zarr(path_label, consolidated=True))
        eyes.append(xr.open_zarr(path_eye, consolidated=True))
    
    # Nối các năm lại với nhau theo trục thời gian
    ds_x = xr.concat(xs, dim='time')
    ds_label = xr.concat(labels, dim='time')
    ds_eye = xr.concat(eyes, dim='time')
    return ds_x, ds_label, ds_eye

# Khởi tạo Train và Val
ds_train_x, ds_train_label, ds_eye_train = load_split(TRAIN_YEARS)
ds_val_x, ds_val_label, ds_eye_val       = load_split(VAL_YEARS)

lats = ds_train_x.lat.values
lons = ds_train_x.lon.values
ALL_VARS = list(ds_train_x.data_vars)

# ── Tách dynamic vs static ──────────────────────────────────────
DYNAMIC_VARS = [v for v in ALL_VARS if v not in STATIC_VARS]
STATIC_VARS_PRESENT = [v for v in STATIC_VARS if v in ALL_VARS]

print(f"  All ERA5 vars  : {ALL_VARS}")
print(f"  Dynamic (Y)    : {DYNAMIC_VARS}")
print(f"  Static (X only): {STATIC_VARS_PRESENT}")
print(f"  Grid           : {len(lats)} lat × {len(lons)} lon = {len(lats)*len(lons)}")
print(f"  Train samples  : {len(ds_train_x.time)} (từ {len(TRAIN_YEARS)} năm)")
print(f"  Val samples    : {len(ds_val_x.time)} (từ {len(VAL_YEARS)} năm)")

--- Load multi-year data (lazy & auto-search Kaggle input) ---
  All ERA5 vars  : ['d2m', 'geopotential_surf', 'lsm', 'msl', 'q', 't2m', 'tp', 'u', 'u10', 'v', 'v10', 'z']
  Dynamic (Y)    : ['d2m', 'msl', 'q', 't2m', 'tp', 'u', 'u10', 'v', 'v10', 'z']
  Static (X only): ['lsm', 'geopotential_surf']
  Grid           : 159 lat × 201 lon = 31959
  Train samples  : 7303 (từ 5 năm)
  Val samples    : 1459 (từ 1 năm)


In [4]:
# ════════════════════════════════════════════════════════════════════
#  TIỀN-GIẢI-NÉN Zarr → memmap fp16 (CHẠY 1 LẦN cho mỗi split)
#  ─ Lý do: Zarr được lưu với chunk {'time':100,...} nên đọc lẻ 1 timestep
#    phải giải nén cả khối 100 bước → thừa ~100×, lặp lại mỗi epoch.
#  ─ Cách: quét tuần tự theo khối (mỗi chunk giải nén đúng 1 lần), ghi fp16
#    ra ổ đĩa. Sau đó Dataset chỉ index → không giải nén lại, không OOM.
# ════════════════════════════════════════════════════════════════════
print("--- Materialize splits → fp16 memmap ---")

def _expand_channels(var_list, ds):
    ch = []
    for v in var_list:
        if v not in ds.data_vars: continue
        da = ds[v]
        if 'pressure_level' in da.dims:
            for lev in da.pressure_level.values:
                ch.append((v, int(lev)))
        else:
            ch.append((v, None))
    return ch

def _read_block(ds, channels, t0, t1):
    """Đọc [t0:t1) cho mọi kênh → (blk, n_grid, C) fp32 (đã nan_to_num)."""
    blk = t1 - t0
    arrs = []
    for v, lev in channels:
        da = ds[v].isel(time=slice(t0, t1))
        if lev is not None:
            da = da.sel(pressure_level=lev)
        a = da.values.astype(np.float32).reshape(blk, -1)   # decompress 1 lần
        arrs.append(a)
    out = np.stack(arrs, axis=-1)                            # (blk, n_grid, C)
    return np.nan_to_num(out, copy=False)

def materialize_split(ds_x, ds_label, ds_eye, dyn_vars, lats, lons, tag):
    n_grid = len(lats) * len(lons)
    dyn_ch = _expand_channels(dyn_vars, ds_x)
    n_dyn  = len(dyn_ch)
    T      = len(ds_x.time)

    paths = {
        'dyn':   f"{SCRATCH}/{tag}_dyn_{T}x{n_grid}x{n_dyn}.f16",
        'label': f"{SCRATCH}/{tag}_label_{T}x{n_grid}x{n_dyn}.f16",
        'tb':    f"{SCRATCH}/{tag}_tb_{T}x{n_grid}.f16",
        'mask':  f"{SCRATCH}/{tag}_mask_{T}x{n_grid}.f16",
    }
    done_flag = f"{SCRATCH}/{tag}_DONE_{T}_{n_grid}_{n_dyn}"

    if os.path.exists(done_flag):
        print(f"  [{tag}] đã có cache (T={T}, C={n_dyn}) → bỏ qua decode.")
    else:
        dyn_mm   = np.memmap(paths['dyn'],   dtype=np.float16, mode='w+', shape=(T, n_grid, n_dyn))
        label_mm = np.memmap(paths['label'], dtype=np.float16, mode='w+', shape=(T, n_grid, n_dyn))
        tb_mm    = np.memmap(paths['tb'],    dtype=np.float16, mode='w+', shape=(T, n_grid))
        mask_mm  = np.memmap(paths['mask'],  dtype=np.float16, mode='w+', shape=(T, n_grid))

        for t0 in range(0, T, MM_BLOCK):
            t1 = min(t0 + MM_BLOCK, T)
            dyn_mm[t0:t1]   = _read_block(ds_x,     dyn_ch, t0, t1).astype(np.float16)
            label_mm[t0:t1] = _read_block(ds_label, dyn_ch, t0, t1).astype(np.float16)
            tb_raw = ds_eye['Tb'].isel(time=slice(t0, t1)).values.astype(np.float32).reshape(t1 - t0, -1)
            mask_mm[t0:t1] = (~np.isnan(tb_raw)).astype(np.float16)
            tb_mm[t0:t1]   = np.nan_to_num(tb_raw, nan=0.0, copy=False).astype(np.float16)
            print(f"    [{tag}] {t1}/{T}", end='\r', flush=True)

        dyn_mm.flush(); label_mm.flush(); tb_mm.flush(); mask_mm.flush()
        del dyn_mm, label_mm, tb_mm, mask_mm
        open(done_flag, 'w').close()
        gb = (T * n_grid * (n_dyn * 2 + 1)) * 2 / 1e9
        print(f"\n  [{tag}] xong. ~{gb:.1f} GB fp16 trên đĩa, {n_dyn} kênh động lực.")

    # ── static (lsm/geopot + sin/cos lat-lon) — nhỏ, giữ trong RAM ──
    static_ch = _expand_channels(STATIC_VARS_PRESENT, ds_x)
    if static_ch:
        ds0 = ds_x.isel(time=0)
        arrs = []
        for v, lev in static_ch:
            da = ds0[v]
            if lev is not None: da = da.sel(pressure_level=lev)
            arrs.append(da.values.flatten())
        stat_e5 = np.stack(arrs, axis=1).astype(np.float32)
    else:
        stat_e5 = np.zeros((n_grid, 0), dtype=np.float32)
    lat_r, lon_r = np.deg2rad(lats), np.deg2rad(lons)
    lat_g, lon_g = np.meshgrid(lat_r, lon_r, indexing='ij')
    pos = np.stack([np.sin(lat_g), np.cos(lat_g),
                    np.sin(lon_g), np.cos(lon_g)], axis=-1).reshape(n_grid, 4).astype(np.float32)
    static = np.nan_to_num(np.concatenate([stat_e5, pos], axis=1), copy=False)

    return {'paths': paths, 'T': T, 'n_grid': n_grid, 'n_dyn': n_dyn,
            'static': torch.from_numpy(static)}

train_mm = materialize_split(ds_train_x, ds_train_label, ds_eye_train, DYNAMIC_VARS, lats, lons, 'train')
val_mm   = materialize_split(ds_val_x,   ds_val_label,   ds_eye_val,   DYNAMIC_VARS, lats, lons, 'val')
print("✓ Materialize hoàn tất.")


--- Materialize splits → fp16 memmap ---
    [train] 7303/7303
  [train] xong. ~13.5 GB fp16 trên đĩa, 14 kênh động lực.
    [val] 1459/1459
  [val] xong. ~2.7 GB fp16 trên đĩa, 14 kênh động lực.
✓ Materialize hoàn tất.


## 4.2 Dataset (2-timestep input, static-as-feature, NaN mask)

In [5]:
class WeatherDatasetMM(Dataset):
    """Đọc từ memmap fp16 đã giải nén sẵn → __getitem__ chỉ là indexing.
       Không giải nén Zarr, không OOM (memmap backed-by-disk, OS tự phân trang)."""
    def __init__(self, mm):
        self.T       = mm['T']
        self.n_grid  = mm['n_grid']
        self.n_dyn   = mm['n_dyn']
        self.paths   = mm['paths']
        self.static  = mm['static']                 # (n_grid, n_static) fp32
        self.n_static_total = self.static.shape[1]
        self.n_in  = self.n_dyn * 2 + 4 + self.n_static_total
        self.n_out = self.n_dyn
        self._mm = None                              # mở lười trong từng worker
        print(f"    n_in={self.n_in} (2×{self.n_dyn} dyn + 4 nasa + {self.n_static_total} static) | n_out={self.n_out}")

    def _ensure_open(self):
        if self._mm is None:
            T, n_grid, n_dyn = self.T, self.n_grid, self.n_dyn
            self._mm = {
                'dyn':   np.memmap(self.paths['dyn'],   dtype=np.float16, mode='r', shape=(T, n_grid, n_dyn)),
                'label': np.memmap(self.paths['label'], dtype=np.float16, mode='r', shape=(T, n_grid, n_dyn)),
                'tb':    np.memmap(self.paths['tb'],    dtype=np.float16, mode='r', shape=(T, n_grid)),
                'mask':  np.memmap(self.paths['mask'],  dtype=np.float16, mode='r', shape=(T, n_grid)),
            }

    def __len__(self):
        return self.T - 1

    def __getitem__(self, idx):
        self._ensure_open()
        t = idx + 1
        mm = self._mm
        dyn_t   = torch.from_numpy(np.asarray(mm['dyn'][t],     dtype=np.float32))
        dyn_tm1 = torch.from_numpy(np.asarray(mm['dyn'][t - 1], dtype=np.float32))
        y       = torch.from_numpy(np.asarray(mm['label'][t],   dtype=np.float32))
        tb_t    = torch.from_numpy(np.asarray(mm['tb'][t],      dtype=np.float32)).unsqueeze(1)
        tb_tm1  = torch.from_numpy(np.asarray(mm['tb'][t - 1],  dtype=np.float32)).unsqueeze(1)
        mk_t    = torch.from_numpy(np.asarray(mm['mask'][t],    dtype=np.float32)).unsqueeze(1)
        mk_tm1  = torch.from_numpy(np.asarray(mm['mask'][t - 1],dtype=np.float32)).unsqueeze(1)

        x = torch.cat([dyn_t, dyn_tm1, tb_t, mk_t, tb_tm1, mk_tm1, self.static], dim=-1)
        return x, y


train_dataset = WeatherDatasetMM(train_mm)
val_dataset   = WeatherDatasetMM(val_mm)

assert train_dataset.n_in  == val_dataset.n_in
assert train_dataset.n_out == val_dataset.n_out
N_IN, N_OUT, N_DYN = train_dataset.n_in, train_dataset.n_out, train_dataset.n_dyn

_loader_kw = dict(num_workers=NUM_WORKERS, pin_memory=True)
if NUM_WORKERS > 0:
    _loader_kw.update(persistent_workers=True, prefetch_factor=PREFETCH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  **_loader_kw)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, **_loader_kw)

print(f"\n✓ Train: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"  Val  : {len(val_dataset)} samples, {len(val_loader)} batches")


    n_in=38 (2×14 dyn + 4 nasa + 6 static) | n_out=14
    n_in=38 (2×14 dyn + 4 nasa + 6 static) | n_out=14

✓ Train: 7302 samples, 609 batches
  Val  : 1458 samples, 122 batches


## Model — U-Net 2D (residual, nhận (B, n_grid, n_in))

In [6]:
import torch.nn.functional as F

class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1), nn.GroupNorm(8, cout), nn.SiLU(),
            nn.Conv2d(cout, cout, 3, padding=1), nn.GroupNorm(8, cout), nn.SiLU())
    def forward(self, x): return self.net(x)


class UNetSEA(nn.Module):
    """Cùng giao diện với GraphCast-SEA: forward chỉ nhận x_grid (B, n_grid, n_in),
       trả (B, n_grid, n_out) theo công thức residual ΔX (so sánh công bằng)."""
    def __init__(self, n_in, n_out, H, W, base=48, depth=3):
        super().__init__()
        self.n_in, self.n_out, self.H, self.W, self.depth = n_in, n_out, H, W, depth
        mult = 2 ** depth
        self.Hp = ((H + mult - 1) // mult) * mult
        self.Wp = ((W + mult - 1) // mult) * mult
        chs = [base * (2 ** i) for i in range(depth + 1)]

        self.inc   = DoubleConv(n_in, chs[0])
        self.downs = nn.ModuleList([DoubleConv(chs[i], chs[i + 1]) for i in range(depth)])
        self.pool  = nn.MaxPool2d(2)
        self.upconvs = nn.ModuleList([nn.ConvTranspose2d(chs[i], chs[i - 1], 2, stride=2)
                                      for i in range(depth, 0, -1)])
        self.ups     = nn.ModuleList([DoubleConv(chs[i], chs[i - 1]) for i in range(depth, 0, -1)])
        self.outc = nn.Conv2d(chs[0], n_out, 1)
        nn.init.zeros_(self.outc.weight); nn.init.zeros_(self.outc.bias)

    def forward(self, x_grid):
        B = x_grid.size(0)
        residual_base = x_grid[..., :self.n_out]
        x = x_grid.permute(0, 2, 1).reshape(B, self.n_in, self.H, self.W)
        x = F.pad(x, (0, self.Wp - self.W, 0, self.Hp - self.H), mode='replicate')

        skips = []
        h = self.inc(x); skips.append(h)
        for down in self.downs:
            h = down(self.pool(h)); skips.append(h)
        h = skips[-1]
        for idx, (upc, up) in enumerate(zip(self.upconvs, self.ups)):
            h = upc(h)
            h = up(torch.cat([h, skips[-(idx + 2)]], dim=1))
        out = self.outc(h)[:, :, :self.H, :self.W]
        delta = out.reshape(B, self.n_out, self.H * self.W).permute(0, 2, 1)
        return residual_base + delta


H_GRID, W_GRID = len(lats), len(lons)
model = UNetSEA(N_IN, N_OUT, H_GRID, W_GRID, base=UNET_BASE, depth=UNET_DEPTH).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ U-Net: {n_params:,} params | grid {H_GRID}×{W_GRID} → pad {model.Hp}×{model.Wp}")
print(f"  N_IN={N_IN}, N_OUT={N_OUT} | residual ΔX | base={UNET_BASE} depth={UNET_DEPTH}")


✓ U-Net: 4,350,686 params | grid 159×201 → pad 160×208
  N_IN=38, N_OUT=14 | residual ΔX | base=48 depth=3


## 4.4 Training

In [7]:
# ── Latitude-weighted RMSE loss ───────────────────────────────────
lat_weights = torch.tensor(np.cos(np.deg2rad(lats)), dtype=torch.float32)
lat_weights = lat_weights / lat_weights.mean()
lat_w_grid = lat_weights.unsqueeze(1).expand(-1, len(lons)).reshape(-1).to(DEVICE)

def weighted_rmse(pred, target):
    se = (pred.float() - target.float()) ** 2          # loss tính ở fp32 cho ổn định
    w  = lat_w_grid.unsqueeze(0).unsqueeze(-1)
    return torch.sqrt((se * w).mean())

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
scaler    = torch.amp.GradScaler('cuda', enabled=USE_AMP)

# ── Multi-GPU (DataParallel) ──────────────────────────────────────
# Graph đã là buffer trong model nên DataParallel tự nhân bản sang mỗi GPU;
# forward chỉ nhận x (chia theo batch). Lưu/khôi phục qua model_core.
n_gpu = torch.cuda.device_count()
if USE_MULTI_GPU and n_gpu > 1:
    net = torch.nn.DataParallel(model)
    print(f"✓ DataParallel trên {n_gpu} GPU (batch {BATCH_SIZE} → {BATCH_SIZE // n_gpu}/GPU)")
else:
    net = model
    print(f"✓ Chạy 1 GPU" if n_gpu else "✓ Chạy CPU")
model_core = net.module if isinstance(net, torch.nn.DataParallel) else net

def run_model(x):
    return net(x)

# ── Persistence baseline ──────────────────────────────────────────
print("--- Persistence baseline (val) ---")
with torch.no_grad():
    losses = []
    for x_batch, y_batch in val_loader:
        x_batch = x_batch.to(DEVICE, non_blocking=True)
        y_batch = y_batch.to(DEVICE, non_blocking=True)
        losses.append(weighted_rmse(x_batch[..., :N_OUT], y_batch).item())
    persist_val = float(np.mean(losses))
print(f"  Persistence val loss: {persist_val:.4f}")

# ── Training loop ─────────────────────────────────────────────────
print("\n--- Bắt đầu training ---")
history = {'train_loss': [], 'val_loss': [], 'lr': [], 'persistence_val': persist_val}
best_val = float('inf'); patience_cnt = 0

for epoch in range(1, N_EPOCHS + 1):
    net.train()
    train_losses = []
    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(DEVICE, non_blocking=True)
        y_batch = y_batch.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            pred = run_model(x_batch)
            loss = weighted_rmse(pred, y_batch)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model_core.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        train_losses.append(loss.item())

    net.eval()
    val_losses = []
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch = x_batch.to(DEVICE, non_blocking=True)
            y_batch = y_batch.to(DEVICE, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                pred = run_model(x_batch)
                vl = weighted_rmse(pred, y_batch)
            val_losses.append(vl.item())

    train_loss = float(np.mean(train_losses)); val_loss = float(np.mean(val_losses))
    lr_now = scheduler.get_last_lr()[0]; scheduler.step()
    history['train_loss'].append(train_loss); history['val_loss'].append(val_loss); history['lr'].append(lr_now)

    beats = "✓" if val_loss < persist_val else "✗"
    print(f"Epoch {epoch:3d}/{N_EPOCHS} | train={train_loss:.4f} | val={val_loss:.4f} "
          f"(persist={persist_val:.4f} {beats}) | lr={lr_now:.2e}")

    if val_loss < best_val:
        best_val = val_loss; patience_cnt = 0
        torch.save({
            'epoch': epoch, 'model_state': model_core.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'val_loss': best_val, 'persistence_val': persist_val,
            'config': {'n_in': N_IN, 'n_out': N_OUT, 'n_dyn': N_DYN,
                       'model_type': 'unet', 'unet_base': UNET_BASE, 'unet_depth': UNET_DEPTH,
                       'grid_hw': [len(lats), len(lons)],
                       'dynamic_vars': DYNAMIC_VARS, 'static_vars': STATIC_VARS_PRESENT,
                       'n_past': 2, 'residual': True},
        }, f'{OUT}/unet_baseline_best.pt')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f"  Early stopping at epoch {epoch}"); break

with open(f'{OUT}/unet_training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f"\n✅ Training xong! Best val={best_val:.4f} | Persistence={persist_val:.4f} "
      f"| Improvement={(1-best_val/persist_val)*100:+.1f}%")
print(f"   Checkpoint: {OUT}/unet_baseline_best.pt")
print(f"   GPUs used: {n_gpu if (USE_MULTI_GPU and n_gpu>1) else 1}")


✓ DataParallel trên 2 GPU (batch 12 → 6/GPU)
--- Persistence baseline (val) ---
  Persistence val loss: 0.5465

--- Bắt đầu training ---
Epoch   1/50 | train=0.4234 | val=0.3889 (persist=0.5465 ✓) | lr=1.50e-04
Epoch   2/50 | train=0.3595 | val=0.3578 (persist=0.5465 ✓) | lr=1.50e-04
Epoch   3/50 | train=0.3373 | val=0.3408 (persist=0.5465 ✓) | lr=1.49e-04
Epoch   4/50 | train=0.3250 | val=0.3315 (persist=0.5465 ✓) | lr=1.49e-04
Epoch   5/50 | train=0.3170 | val=0.3263 (persist=0.5465 ✓) | lr=1.48e-04
Epoch   6/50 | train=0.3102 | val=0.3200 (persist=0.5465 ✓) | lr=1.46e-04
Epoch   7/50 | train=0.3046 | val=0.3153 (persist=0.5465 ✓) | lr=1.45e-04
Epoch   8/50 | train=0.3004 | val=0.3140 (persist=0.5465 ✓) | lr=1.43e-04
Epoch   9/50 | train=0.2967 | val=0.3119 (persist=0.5465 ✓) | lr=1.41e-04
Epoch  10/50 | train=0.2930 | val=0.3100 (persist=0.5465 ✓) | lr=1.38e-04
Epoch  11/50 | train=0.2899 | val=0.3082 (persist=0.5465 ✓) | lr=1.36e-04
Epoch  12/50 | train=0.2870 | val=0.3075 (persist

## Đánh giá — Rollout RMSE vs Persistence (TEST) để so với GraphCast

In [8]:
# Nạp + materialize TEST (2026) để đánh giá trên CÙNG split với nb5 (GraphCast).
ds_test_x, ds_test_label, ds_eye_test = load_split(TEST_YEARS)
test_mm = materialize_split(ds_test_x, ds_test_label, ds_eye_test, DYNAMIC_VARS, lats, lons, 'test')

te_dyn  = np.memmap(test_mm['paths']['dyn'],  dtype=np.float16, mode='r',
                    shape=(test_mm['T'], test_mm['n_grid'], test_mm['n_dyn']))
te_tb   = np.memmap(test_mm['paths']['tb'],   dtype=np.float16, mode='r', shape=(test_mm['T'], test_mm['n_grid']))
te_mask = np.memmap(test_mm['paths']['mask'], dtype=np.float16, mode='r', shape=(test_mm['T'], test_mm['n_grid']))
te_static = test_mm['static'].numpy()
T_test    = test_mm['T']

net.eval()

def assemble_input(dyn_t, dyn_tm1, t_idx):
    tb_t   = te_tb[t_idx].astype(np.float32).reshape(-1, 1)
    mk_t   = te_mask[t_idx].astype(np.float32).reshape(-1, 1)
    tb_tm1 = te_tb[t_idx - 1].astype(np.float32).reshape(-1, 1)
    mk_tm1 = te_mask[t_idx - 1].astype(np.float32).reshape(-1, 1)
    x = np.concatenate([dyn_t, dyn_tm1, tb_t, mk_t, tb_tm1, mk_tm1, te_static], axis=-1)
    return torch.from_numpy(x).unsqueeze(0).float().to(DEVICE)

def rollout(start_idx, n_steps):
    dyn_t   = te_dyn[start_idx].astype(np.float32)
    dyn_tm1 = te_dyn[start_idx - 1].astype(np.float32)
    preds, gts = [], []
    with torch.no_grad():
        for step in range(n_steps):
            t_now = start_idx + step
            if t_now + 1 >= T_test: break
            x = assemble_input(dyn_t, dyn_tm1, t_now)
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                pred = net(x)
            p = pred.squeeze(0).float().cpu().numpy()
            preds.append(p)
            gts.append(te_dyn[t_now + 1].astype(np.float32))   # state thực ở t+1
            dyn_tm1, dyn_t = dyn_t, p
    return preds, gts

# Multi-start (trùng cấu hình nb5: 30 starts × 12 steps)
N_STEPS  = 12
N_STARTS = 30
MAX_START = T_test - N_STEPS - 1
start_indices = np.linspace(1, MAX_START, N_STARTS, dtype=int)

lat_w = np.repeat(np.cos(np.deg2rad(lats)) / np.cos(np.deg2rad(lats)).mean(), len(lons))
rmse_u = np.zeros((N_STEPS, N_DYN)); rmse_p = np.zeros((N_STEPS, N_DYN)); cnt = np.zeros(N_STEPS)

print(f"--- U-Net rollout: {N_STARTS} starts × {N_STEPS} steps (TEST {TEST_YEARS}) ---")
for si in start_indices:
    preds, gts = rollout(si, N_STEPS)
    base = te_dyn[si].astype(np.float32)
    for step, (p, g) in enumerate(zip(preds, gts)):
        for v in range(N_DYN):
            rmse_u[step, v] += ((p[:, v] - g[:, v]) ** 2 * lat_w).mean()
            rmse_p[step, v] += ((base[:, v] - g[:, v]) ** 2 * lat_w).mean()
        cnt[step] += 1
rmse_u = np.sqrt(rmse_u / cnt[:, None]); rmse_p = np.sqrt(rmse_p / cnt[:, None])

# Physical units (×std) — cùng cách nb5
import json as _json
try:
    with open(STATS_JSON) as f: stats = _json.load(f)
    dyn_ch = []
    for vv in DYNAMIC_VARS:
        if vv not in ds_test_x.data_vars: continue
        da = ds_test_x[vv]
        if 'pressure_level' in da.dims:
            for lev in da.pressure_level.values: dyn_ch.append((vv, int(lev)))
        else: dyn_ch.append((vv, None))
    def _std(v, l):
        k = v if l is None else f"{v}_{l}hpa"
        return stats[k]['std'] if k in stats else next((stats[kk]['std'] for kk in stats if kk.startswith(v)), 1.0)
    stds = np.array([_std(v, l) for v, l in dyn_ch])
except Exception as e:
    print("  (bỏ qua đơn vị vật lý:", e, ")"); dyn_ch = [(f"ch{i}", None) for i in range(N_DYN)]; stds = np.ones(N_DYN)

rmse_u_phys = rmse_u * stds[None, :]; rmse_p_phys = rmse_p * stds[None, :]

print(f"\n{'var':<22s}  +6h(U/P)        +24h            +48h            +72h")
for vi, (vn, lev) in enumerate(dyn_ch):
    tag = vn if lev is None else f"{vn}@{lev}"
    row = ""
    for s in [0, 3, 7, 11]:
        if s < N_STEPS:
            u, pp = rmse_u_phys[s, vi], rmse_p_phys[s, vi]
            row += f"  {u:.3f}/{pp:.3f}{'✓' if u < pp else '✗'}"
    print(f"  {tag:<20s}{row}")

# Lưu (split=test, cùng format nb5 để ghép bảng so sánh)
summary = {'model': 'unet', 'split': 'test', 'lead_hours': (np.arange(1, N_STEPS+1)*6).tolist(),
           'n_starts': int(N_STARTS),
           'channels': [{'var': v, 'level': l} for v, l in dyn_ch],
           'rmse_unet_normalized': rmse_u.tolist(),
           'rmse_persist_normalized': rmse_p.tolist(),
           'rmse_unet_physical': rmse_u_phys.tolist(),
           'rmse_persist_physical': rmse_p_phys.tolist()}
with open(f'{OUT}/unet_baseline_rmse.json', 'w') as f: _json.dump(summary, f, indent=2)
print(f"\n✓ Saved: {OUT}/unet_baseline_rmse.json (TEST)")
print("  → Cùng split với nb5: đối chiếu rmse_unet_* với rmse_model_* của GraphCast.")
